# Aumento de datos en CIFAR-10: comparación controlada

Se reproduce la arquitectura y el protocolo de entrenamiento del Momento 3. La única variable experimental es el bloque de aumento de datos dentro del modelo. El conjunto de prueba oficial permanece intacto. **El aumento de datos no reemplaza datos representativos.**

## Configuración

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.environ["MPLCONFIGDIR"] = str(ROOT / ".cache" / "matplotlib")
os.environ["XDG_CACHE_HOME"] = str(ROOT / ".cache")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

SCENARIOS = ("reference", "full", "flip_h", "rotation", "zoom", "flip_v", "blur")
SEEDS = (42,)
EPOCHS = 30
SUBSET_PER_CLASS = None  # Set to 100 for a balanced training-only smoke run.
DATA_DIR = ROOT / "data" / "cifar-10-batches-py"
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)


## Exploración de datos

Se utilizan exclusivamente los archivos locales de la versión Python de CIFAR-10; no se realiza ninguna descarga. El control de duplicados próximos mediante pHash identifica **candidatos**, no confirma duplicación semántica ni fuga de información. Se examinan los conjuntos oficiales completos de entrenamiento y prueba.

In [ ]:
from src.data import load_cifar10, make_splits, normalize, dataset_summary, image_statistics, near_duplicate_train_test
from src.plotting import plot_class_balance, plot_image_statistics, plot_class_samples, plot_near_duplicate_examples

train_raw, train_labels_full, test_raw, test_labels, class_names = load_cifar10(DATA_DIR)
splits = make_splits(train_labels_full, RESULTS / "splits.npz", test_count=len(test_raw))
summary = dataset_summary(train_labels_full, test_labels, splits, RESULTS / "dataset_summary.csv")
display(summary)
print("Image resolution:", train_raw.shape[1:], "| train:", len(train_raw), "| test:", len(test_raw))
plot_class_balance(summary, FIGURES / "class_balance.png")
plot_class_samples(train_raw[splits["train"]], train_labels_full[splits["train"]], FIGURES / "class_samples.png")
train_stats = image_statistics(train_raw[splits["train"]], train_labels_full[splits["train"]])
display(train_stats.groupby("class")[["mean_brightness", "contrast_std"]].agg(["mean", "std"]))
plot_image_statistics(train_stats, FIGURES / "brightness_contrast.png")
for name in ("class_balance.png", "class_samples.png", "brightness_contrast.png"):
    display(Image(filename=str(FIGURES / name)))


In [ ]:
duplicate_screen = near_duplicate_train_test(train_raw, test_raw, max_hamming_distance=4)
print("pHash screen (full official train/test sets):", {key: value for key, value in duplicate_screen.items() if key != "examples"})
plot_near_duplicate_examples(train_raw, test_raw, duplicate_screen["examples"], FIGURES / "near_duplicate_candidates.png")
display(Image(filename=str(FIGURES / "near_duplicate_candidates.png")))


### Analisis (escribir por el estudiante)

- ¿Las particiones conservan el balance por clase?
- ¿Qué diferencias de brillo y contraste se observan?
- ¿Cuántos pares candidatos detecta pHash y qué limitaciones tiene este criterio?
- ¿Qué clases parecen visualmente más difíciles a 32 × 32 píxeles?

## Protocolo experimental

La partición estratificada del 80/20 usa `random_state=42`. La normalización divide por 255 en `float32`. El subconjunto opcional afecta únicamente al entrenamiento: validación y prueba permanecen completos. Las transformaciones se aplican dentro del modelo solo durante entrenamiento. La precisión mixta del entorno Colab del Momento 3 se omite en esta ejecución local.

In [ ]:
from src.augmentation import build_augmentation, assert_train_only
from src.model import build_cnn_model
from src.experiment import run_experiments, balanced_subset_indices
from src.plotting import plot_augmentations, plot_learning_curves, plot_confusions, plot_per_class_f1, plot_delta_f1, plot_prediction_examples

x_train = normalize(train_raw[splits["train"]])
y_train = train_labels_full[splits["train"]]
x_val = normalize(train_raw[splits["validation"]])
y_val = train_labels_full[splits["validation"]]
x_test = normalize(test_raw)
y_test = test_labels

config = pd.DataFrame([
    ("Scenarios", ", ".join(SCENARIOS)), ("Seeds", str(SEEDS)), ("Epoch limit", EPOCHS),
    ("Train subset per class", str(SUBSET_PER_CLASS)), ("Official train/val/test", f"{len(x_train)}/{len(x_val)}/{len(x_test)}"),
    ("Batch size", 128), ("Optimizer", "Adam, learning rate 1e-3"),
    ("Loss", "sparse_categorical_crossentropy"),
    ("EarlyStopping", "val_loss, patience=6, restore_best_weights=True"),
    ("ReduceLROnPlateau", "val_loss, factor=0.5, patience=2, min_lr=1e-6"),
    ("tf.data shuffle", "buffer_size=10000, seed per run"), ("Mixed precision", "disabled"),
], columns=["parameter", "value"])
display(config)
model = build_cnn_model(build_augmentation("full", seed=SEEDS[0]))
model.summary()
print("Model parameters:", model.count_params())
assert model.count_params() == 290090


In [ ]:
example_indices = [int(np.flatnonzero(y_train == class_id)[0]) for class_id in range(6)]
example_images = x_train[example_indices]
for scenario in SCENARIOS:
    augmenter = build_augmentation(scenario, seed=SEEDS[0])
    candidate = build_cnn_model(augmenter)
    assert_train_only(candidate, augmenter, example_images)
print("Training-only/inference identity assertions passed for all scenarios.")
plot_augmentations(x_train, y_train, FIGURES / "augmentation_grid.png", scenarios=SCENARIOS, seed=SEEDS[0])
display(Image(filename=str(FIGURES / "augmentation_grid.png")))


## Entrenamiento y evaluación

Todos los escenarios usan la misma arquitectura, particiones y configuración de entrenamiento. Una ejecución completa se reutiliza solo cuando su manifiesto de configuración coincide (épocas, tamaño de lote, subconjunto e identidad de los datos). Una discrepancia detiene el proceso; los resultados incompletos se recalculan.

In [ ]:
metrics, gaps, per_class = run_experiments(
    x_train, y_train, x_val, y_val, x_test, y_test,
    scenarios=SCENARIOS, seeds=SEEDS, epochs=EPOCHS,
    subset_per_class=SUBSET_PER_CLASS, output_root=RESULTS,
)
display(metrics)
display(gaps)
display(per_class)


In [ ]:
if {"reference", "full"}.issubset(set(SCENARIOS)):
    plot_learning_curves(RESULTS, SEEDS[0], FIGURES / "learning_curves.png")
    plot_confusions(RESULTS, SEEDS[0], FIGURES / "confusion_matrices.png")
    plot_delta_f1(per_class, FIGURES / "delta_f1.png")
    for name in ("learning_curves.png", "confusion_matrices.png", "delta_f1.png"):
        display(Image(filename=str(FIGURES / name)))
plot_per_class_f1(per_class, FIGURES / "per_class_f1.png")
display(Image(filename=str(FIGURES / "per_class_f1.png")))
for scenario in ("reference", "full"):
    if scenario in SCENARIOS:
        name = f"prediction_examples_{scenario}.png"
        plot_prediction_examples(x_test, RESULTS, scenario, SEEDS[0], FIGURES / name)
        display(Image(filename=str(FIGURES / name)))


### Analisis (escribir por el estudiante)

- ¿Qué diferencia de exactitud y F1 macro presenta el aumento completo frente a la referencia?
- ¿Cómo cambian las curvas de entrenamiento y validación y la brecha de generalización?
- ¿Qué clases mejoran o empeoran en F1 y qué errores persisten en las matrices de confusión?
- ¿Qué muestran los ejemplos de predicciones de alta confianza?

### Analisis (escribir por el estudiante): ablaciones y pruebas

- ¿Cuál transformación individual aporta más y cuál perjudica el desempeño?
- ¿El reflejo vertical y el desenfoque conservan razonablemente la etiqueta en este conjunto?
- ¿Las diferencias se mantienen entre semillas o pueden deberse a variabilidad experimental?

## Conclusiones y limitaciones

- ¿Qué conclusión respaldan directamente las métricas observadas?
- ¿Qué limita la interpretación: resolución, semillas, presupuesto de entrenamiento, duplicados candidatos o diseño de transformaciones?
- ¿Qué experimento adicional permitiría poner a prueba la explicación propuesta?

El aumento de datos no reemplaza un conjunto de datos representativo.